<a href="https://colab.research.google.com/github/sayurireddy/SReddy-DataScience-GenAI-Submissions/blob/main/Assignment_2/2_02_Joining_DataFrames.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=1xqQczl0FG-qtNA2_WQYuWePW9oU8irqJ)

# 2.02 Pandas: Joining and merging dataframes
### Why Combine Dataframes?
Quite often in practical applications we have data coming from multiple sources such as different files, database tables or IoT devices. This is particularly likely when working in data science and artificial intelligence. However, typically we want to create a single dataframe to run our analysis and/or models. To achieve this commonly we would want to combine dataframes together.

Joining data is a large topic and an important one in database administration and data engineering. We will look at this at a relatively high level, althout the interested reader can certainly explore this in much more depth.

Let us begin by using the orders_df from the previous session.

In [1]:
import pandas as pd
import numpy as np

orders = {'o10001':{'date':'2024/01/10', 'product':'Hoodie', 'quantity':'1'},
            'o10002':{'date':'2024/01/13', 'product':'Tote bag', 'quantity':'2'},
            'o10003':{'date':'2024/01/14', 'product':'Pencil', 'quantity':'10'},
            'o10004':{'date':'2024/01/15', 'product':'T-shirt', 'quantity':'2'}
}

orders_df = pd.DataFrame(orders)
orders_df

,o10001,o10002,o10003,o10004
date,2024/01/10,2024/01/13,2024/01/14,2024/01/15
product,Hoodie,Tote bag,Pencil,T-shirt
quantity,1,2,10,2


In this scenario, we also have some data about these products from our operations team. The data is currently in a dictionary and can be transformed in the usual fashion:

In [2]:
products = {'123':{'name':'Hoodie', 'cost_price':12.12, 'sale_price':15.00},
            '124':{'name':'Tote bag', 'cost_price':2.15, 'sale_price':9.99},
            '125':{'name':'Pencil', 'cost_price':22.45, 'sale_price':49.99},
            '126':{'name':'T-shirt', 'cost_price':0.45, 'sale_price':2.99},
            '127':{'name':'Honours degree', 'cost_price':0.78, 'sale_price':1.49}
}

products_df = pd.DataFrame(products)
products_df

,123,124,125,126,127
name,Hoodie,Tote bag,Pencil,T-shirt,Honours degree
cost_price,12.12,2.15,22.45,0.45,0.78
sale_price,15.0,9.99,49.99,2.99,1.49


### Joining Dataframes using Merge
What we want to do is join these datasets together. However, currently the Dataframes are both the wrong way round. We would want our rows in orders_df to be the order ID, and the rows in products_df to be the product ID. We can transform the data using the Dataframe.transpose function:

In [10]:
orders_df = orders_df.transpose()
products_df = products_df.transpose()

Transpose basically flips the axes of the Dataframe so the current columns become rows and vice versa. Note, _Dataframe.transpose_ creates a new object by default. We use the "orders_df = ..." element at the start to reassign the variable and make orders_df equal the new, transposed version.

In [5]:
orders_df

,o10001,o10002,o10003,o10004
date,2024/01/10,2024/01/13,2024/01/14,2024/01/15
product,Hoodie,Tote bag,Pencil,T-shirt
quantity,1,2,10,2


With this in place we can join the two Dataframes. There are several decisions that need to be taken when joining two datasets. Firstly, there are multiple ways to join data in pandas but the most appropriate here would seem to be using the _merge( )_ function. These choices will be discussed in more detail later in the notebook. Secondly we need to decide on the element that we will make the join on. In this case, the data that the two dataframes have in common are the product name, labelled “product” in orders_df and as “name” in products_df. We also need to decide the type of join – what do we want to do with rows that are in one set but not in the other? This too is discussed further below. In this case the scenario is to get more details on our orders, so we are joining products_df to orders_df so we will base our rows on the orders_df dataframe. With these choices in mind we get the following code:

In [8]:
orders_df = orders_df.transpose()
products_df = products_df.transpose()
joined_df = orders_df.merge(products_df, how='left', left_on='product', right_on='name')

We’ll unpack this a little more because it is a more complex function. We are creating a new dataframe called joined_df. This will be based on the orders_df being merged with products_df as a ‘left join’. This means we will use the rows of the orders_df not products_df (as orders_df is on the left hand of the command). Finally we need to tell Python which columns we will be joining based on. In this case the left hand dataframe (orders_df) lists the relevant data as “product”, and the right hand (products_df) lists it as “name”.

In [9]:
joined_df

,date,product,quantity,name,cost_price,sale_price
0,2024/01/10,Hoodie,1,Hoodie,12.12,15.0
1,2024/01/13,Tote bag,2,Tote bag,2.15,9.99
2,2024/01/14,Pencil,10,Pencil,22.45,49.99
3,2024/01/15,T-shirt,2,T-shirt,0.45,2.99


There are several types of joins that can be used here. The main types are shown below. The **yellow bit** in each Venn diagram represents the bit we would **keep** if we perform this type of join.

So far we have only looked at a **"left join" (labelled in pandas as "left")**.

It would also be possible to specify an **"inner join" ("inner")**. This means we only keep records (rows) that are in both datasets.

Only the **records in both would be included if we use an "inner join"**.

Finally we could use a **"full join" ("outer")** which means that we **keep all the records** irrespective of whether they are missing in one of the sides.

![](https://drive.google.com/uc?export=view&id=1e8rPwaCqpnrB9-yIcqKOm2sb6VUPGz_g)

Let’s return to our first merged dataframe (joined_df) as this better fits the description of this problem (we care more about orders in this problem than products). One immediate issue is that we have duplicated columns product and name. Let’s tidy this up by dropping the name column:

In [11]:
joined_df = joined_df.drop(['name'], axis=1)
joined_df

,date,product,quantity,cost_price,sale_price
0,2024/01/10,Hoodie,1,12.12,15.0
1,2024/01/13,Tote bag,2,2.15,9.99
2,2024/01/14,Pencil,10,22.45,49.99
3,2024/01/15,T-shirt,2,0.45,2.99


The _drop( )_ function can be used to remove columns or rows. axis=1 indicates that we want to a drop a column(s), axis=0 would mean we were dropping a row(s). Note as well that although we are only dropping one column we still need to present the column name inside a list as this is what the function expects (a list).

### EXERCISE
In the above example we merged using a left join. How would different types of joins effect the final dataframe? Experiment with inner join, right join and outer join. What impact does it have on the data retained? Are the results what you expected them to be?

## **Inner** Join


This means we only keep records (rows) that are in both datasets. Only the **records in both would be included if we use an "inner join"**.



In [14]:
# Re-created from original dictionaries to ensure a clean state
orders_df = pd.DataFrame(orders).transpose()
products_df = pd.DataFrame(products).transpose()

#Inner join and dropped duplicate column
joined_df = orders_df.merge(products_df, how='inner', left_on='product', right_on='name')
joined_df = joined_df.drop(['name'], axis=1)
joined_df

,date,product,quantity,cost_price,sale_price
0,2024/01/10,Hoodie,1,12.12,15.0
1,2024/01/13,Tote bag,2,2.15,9.99
2,2024/01/14,Pencil,10,22.45,49.99
3,2024/01/15,T-shirt,2,0.45,2.99


### **Impact**:

*   **No Lost Orders**: Every product listed in my orders(orders_df) exists in my product registry(products_df). Therefore, I did not lose any rows from the orders list.

*   **Excluded Unsold Products**: The product **"Honours degree"** exists in products_df but was never ordered. Because an inner join only keeps rows where the joining keys exist in both data frames, "Honours degree" is completely excluded from the final table.

Overall, the inner join kept only the products that were actually ordered, leaving out any products in my inventory that had zero sales.


## **Right** Join
A right joint keeps all records from the right dataframe(products_df) and matches them with records from the left dataframe(orders_df)

If there is no matching record in the left dataframe for a given row in the right dataframe, the missing fiels will be filled with NaN values.

In [15]:
orders_df = pd.DataFrame(orders).transpose()
products_df = pd.DataFrame(products).transpose()

#Right join
right_joined_df = orders_df.merge(products_df, how='right', left_on='product', right_on='name')
right_joined_df

,date,product,quantity,name,cost_price,sale_price
0,2024/01/10,Hoodie,1,Hoodie,12.12,15.0
1,2024/01/13,Tote bag,2,Tote bag,2.15,9.99
2,2024/01/14,Pencil,10,Pencil,22.45,49.99
3,2024/01/15,T-shirt,2,T-shirt,0.45,2.99
4,NaN,NaN,NaN,Honours degree,0.78,1.49


Notice that the last row contains "Honours degree" with `NaN` values for `date` and `quantity` because there was no matching order.

### **Impact:**


*   **All products are retained**: Every single product in the registry (products_df), including **"Honours degree"**, which has never been ordered, appear in the final table.
*   **Unsold items have missing order details:** Since nobody ordered the "Honours degree", its date and quantity fields from the order table are filled with NaN



## **Outer Join**


A full outer join keeps **all records** from both the left dataframe (`orders_df`) and the right dataframe (`products_df`). If there are unmatched rows in either table, pandas will fill the missing values with `NaN`.

In [16]:
orders_df = pd.DataFrame(orders).transpose()
products_df = pd.DataFrame(products).transpose()

#Full outer join
outer_joined_df = orders_df.merge(products_df, how='outer', left_on='product', right_on='name')
outer_joined_df

,date,product,quantity,name,cost_price,sale_price
0,NaN,NaN,NaN,Honours degree,0.78,1.49
1,2024/01/10,Hoodie,1,Hoodie,12.12,15.0
2,2024/01/14,Pencil,10,Pencil,22.45,49.99
3,2024/01/15,T-shirt,2,T-shirt,0.45,2.99
4,2024/01/13,Tote bag,2,Tote bag,2.15,9.99


### **Impact:**
* **No information is lost**: We keep all 4 order records as well as all 5 product records.
* **NaN Handling**: Just like the right join in this specific dataset, "Honours degree" appears with `NaN` values for `date` and `quantity` because there are no matching orders for it. If there were orders for items not present in our product list, they would also appear with `NaN` values for `cost_price` and `sale_price`.

### **Are the results what I expected them to be?**

* **Yes, they match expectations:** The inner join successfully filtered out items without transactions (the 'Honours degree'), while the right and outer joins kept everything but introduced expected `NaN` (null) values where data in one of the tables was missing.